# BPS 2020–2026 | Análise exploratória

Notebook de apoio ao Mini-Projeto Avaliativo de Visualização de Dados e Business Intelligence.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('../data/processed/BPS_20_26_Carolina.csv')
df = pd.read_csv(BASE, sep=';', encoding='utf-8-sig', low_memory=False)
df.shape

In [ ]:
df.head()

## Qualidade dos dados

In [ ]:
pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'pct_nulos': (df.isna().mean()*100).round(2)
}).sort_values('pct_nulos', ascending=False).head(20)

## KPIs

In [ ]:
valor_total = df['vl_preco_total'].sum()
quantidade = df['qt_medicamento'].sum()
kpis = {
    'Valor total registrado': valor_total,
    'Quantidade total': quantidade,
    'Registros': len(df),
    'Instituições': df['cnpj_instituicao'].nunique(),
    'Fornecedores': df['cnpj_fornecedor'].nunique(),
    'Preço unitário médio ponderado': valor_total / quantidade
}
kpis

## Evolução anual

In [ ]:
anual = df.groupby('ano_compra').agg(
    registros=('co_seq_bps','size'),
    valor_total=('vl_preco_total','sum'),
    quantidade=('qt_medicamento','sum'),
    instituicoes=('cnpj_instituicao','nunique'),
    fornecedores=('cnpj_fornecedor','nunique')
)
anual['preco_unitario_medio_ponderado'] = anual['valor_total'] / anual['quantidade']
anual

## Rankings

In [ ]:
top_uf = df.groupby('sg_uf')['vl_preco_total'].sum().sort_values(ascending=False).head(10)
top_produtos = df.groupby('ds_item')['vl_preco_total'].sum().sort_values(ascending=False).head(10)
top_fornecedores = df.groupby('no_fornecedor')['vl_preco_total'].sum().sort_values(ascending=False).head(10)
print('UF\n', top_uf)
print('PRODUTOS\n', top_produtos)
print('FORNECEDORES\n', top_fornecedores)

## Validação do preço total

A relação esperada é `preço total = quantidade × preço unitário`. A diferença pode ocorrer por arredondamento de ponto flutuante.

In [ ]:
calculado = df['qt_medicamento'] * df['vl_preco_unitario']
diferenca = (df['vl_preco_total'] - calculado).abs()
print('Maior diferença absoluta:', diferenca.max())
print('Diferenças acima de R$ 0,01:', (diferenca > 0.01).sum())